In [1]:
import os
import numpy as np
import cv2

In [2]:
def pad_and_resize_volume(volume, target_size=(512, 512)):
    """
    Takes a 3D numpy volume, pads each slice to a square to preserve anatomy,
    and then resizes it to the target dimensions.
    """
    num_slices, h, w = volume.shape

    # Create an empty 3D array for the new standardized slices
    standardized_volume = np.zeros((num_slices, target_size[0], target_size[1]), dtype=np.float32)

    for i in range(num_slices):
        slice_2d = volume[i]

        # --- 1. PADDING (Making it Square) ---
        # If the image is not a perfect square, we pad the shorter sides with 0 (black/air)
        if h != w:
            max_dim = max(h, w)
            pad_h = (max_dim - h) // 2
            pad_w = (max_dim - w) // 2

            # Pad the image. cv2.copyMakeBorder adds borders to the top/bottom/left/right
            slice_2d = cv2.copyMakeBorder(
                slice_2d,
                top=pad_h,
                bottom=max_dim - h - pad_h,
                left=pad_w,
                right=max_dim - w - pad_w,
                borderType=cv2.BORDER_CONSTANT,
                value=0.0 # 0.0 represents the normalized background (air)
            )

        # --- 2. RESIZING ---
        # Now that it is a perfect square, we can safely resize without stretching the lungs
        resized_slice = cv2.resize(slice_2d, (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)

        standardized_volume[i] = resized_slice

    return standardized_volume

In [3]:
def standardize_dataset(source_dir, output_dir, target_size=(512, 512)):
    """
    Loops through all .npy files, applies the pad/resize function, and saves them.
    """
    os.makedirs(output_dir, exist_ok=True)

    patient_files = [f for f in os.listdir(source_dir) if f.endswith('.npy')]

    for file_name in patient_files:
        input_path = os.path.join(source_dir, file_name)
        output_path = os.path.join(output_dir, file_name)

        print(f"Standardizing {file_name}...")

        # Load the raw, un-sized 3D volume
        raw_volume = np.load(input_path)

        # Apply the padding and resizing
        clean_volume = pad_and_resize_volume(raw_volume, target_size)

        # Save the perfectly sized volume to the new folder
        np.save(output_path, clean_volume)
        print(f" --> Saved! Final Shape: {clean_volume.shape}")

# --- Example Usage ---
# The folder where you saved your Stage 1 & 2 outputs
input_data_directory = "drive/MyDrive/CovidDataset_Processed/Processed_Data/COVID-S1"

# A new folder for the perfectly sized, AI-ready data
final_data_directory = "drive/MyDrive/Model_Ready_Data/COVID-S1"

# Run it! (You can change to 256x256 here if your GPU memory is low)
standardize_dataset(input_data_directory, final_data_directory, target_size=(512, 512))

Standardizing C002.npy...
 --> Saved! Final Shape: (144, 512, 512)
Standardizing C005.npy...
 --> Saved! Final Shape: (122, 512, 512)
Standardizing C004.npy...
 --> Saved! Final Shape: (125, 512, 512)
Standardizing C003.npy...
 --> Saved! Final Shape: (163, 512, 512)
Standardizing C006.npy...
 --> Saved! Final Shape: (139, 512, 512)
Standardizing C001.npy...
 --> Saved! Final Shape: (143, 512, 512)
Standardizing C007.npy...
 --> Saved! Final Shape: (144, 512, 512)
Standardizing C010.npy...
 --> Saved! Final Shape: (134, 512, 512)
Standardizing C008.npy...
 --> Saved! Final Shape: (122, 512, 512)
Standardizing C009.npy...
 --> Saved! Final Shape: (141, 512, 512)
Standardizing C011.npy...
 --> Saved! Final Shape: (139, 512, 512)
Standardizing C012.npy...
 --> Saved! Final Shape: (144, 512, 512)
Standardizing C014.npy...
 --> Saved! Final Shape: (144, 512, 512)
Standardizing C015.npy...
 --> Saved! Final Shape: (168, 512, 512)
Standardizing C016.npy...
 --> Saved! Final Shape: (143, 512, 

Neural networks absolutely require every single image you feed them to be the exact same size—usually a perfect square like 512x512 or 256x256.

Here is exactly what that code is doing and why it is critical for your pipeline:

### The Problem
If a hospital CT scanner outputs an image that is a rectangle (for example, 512 pixels wide but only 400 pixels tall), you cannot simply force it into a 512x512 box. If you do, the computer will stretch the image vertically.

* A perfectly round Ground Glass Opacity (GGO) will be stretched into an oval.

* The physical area of the lung will change.

* When your AI gets to Stage 5 and tries to calculate the exact percentage of lung damage, its math will be completely wrong because you warped the patient's anatomy!

### What the Code is Doing (The Solution)
The script we just wrote prevents this distortion using a two-step mathematical trick:

1. **Padding** (The Black Borders)
Instead of stretching the 512x400 image, the code looks at the shorter side (the 400 height) and mathematically glues 56 pixels of black space to the top and 56 pixels of black space to the bottom.

The image is now a perfect 512x512 square, but the lung tissue in the center hasn't been stretched or warped at all. The black pixels just represent the empty air around the patient.

2. **Uniform Resizing**
Now that the image is a perfect square, if we need to shrink it (say, to 256x256 because Colab doesn't have enough GPU memory), the script scales the entire square down evenly. The lungs stay in perfect physical proportion.

3. **Batch Saving**
Finally, the script automatically opens every single patient's .npy file, applies this padding and resizing to all their slices, and saves the perfect, AI-ready blocks into a brand new folder called Model_Ready_Data.